In [ ]:
from azure.identity import  InteractiveBrowserCredential

scope = 'https://analysis.windows.net/powerbi/api/.default'

interactive_browser_credential_class = InteractiveBrowserCredential()
access_token_class = interactive_browser_credential_class.get_token(scope)
token_string = access_token_class.token

print(token_string)

In [ ]:
import requests
url = 'https://wabi-yourregionaddress-primary-redirect.analysis.windows.net/metadata/appmodel/apps/8f64a385-cf22-4c0e-adea-81f5dba07b17?requestDataType=7'

headers = {
    "Authorization": f"Bearer {token_string}",
    "Accept": "application/json"
}

#headers = {"Accept":"application/json","Authorization":token}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    response.json()
else:
    print("Error: ", response.status_code)
#response.json()

In [34]:
import pandas as pd
df=response.json()
df2=pd.DataFrame(df)

In [ ]:
# Create a comprehensive dataframe directly from df2
combined_data = []

# Extract base app information from df2
app_info = df2.iloc[0]

# Iterate through appViewDetails
for view in app_info['appViewDetails']:
    view_id = view['id']
    view_key = view['viewKey']
    view_name = view['viewName']
    report_ids = view['reportIds']
    dashboard_ids = view['dashboardIds']
    is_shared = view['contentProviderPermissions']['isSharedToEntireOrganization']
    allow_copy = view['appSettings']['allowCopyContent']
    explore_dataset = view['appSettings']['exploreDataset']
    reshare_dataset = view['appSettings']['reshareDataset']
    
    # Get users from adUserMetadataList
    ad_users = view['contentProviderPermissions']['adUserMetadataList']
    
    # If no users, add a row with NaN for user fields
    if len(ad_users) == 0:
        combined_data.append({
            'providerId': app_info['providerId'],
            'appId': app_info['providerKey'],
            'appDisplayName': app_info['displayText'],
            'viewId': view_id,
            'viewKey': view_key,
            'viewName': view_name,
            'reportIds': report_ids,
            'dashboardIds': dashboard_ids,
            'isSharedToEntireOrganization': is_shared,
            'allowCopyContent': allow_copy,
            'exploreDataset': explore_dataset,
            'reshareDataset': reshare_dataset,
            'displayName': None,
            'userPrincipalName': None,
            'objectId': None,
            'emailAddress': None
        })
    else:
        # Add a row for each user
        for user in ad_users:
            combined_data.append({
                'providerId': app_info['providerId'],
                'appId': app_info['providerKey'],
                'appDisplayName': app_info['displayText'],
                'viewId': view_id,
                'viewKey': view_key,
                'viewName': view_name,
                'reportIds': report_ids,
                'dashboardIds': dashboard_ids,
                'isSharedToEntireOrganization': is_shared,
                'allowCopyContent': allow_copy,
                'exploreDataset': explore_dataset,
                'reshareDataset': reshare_dataset,
                'displayName': user.get('displayName'),
                'userPrincipalName': user.get('userPrincipalName'),
                'objectId': user.get('objectId'),
                'emailAddress': user.get('emailAddress')
            })

df_combined = pd.DataFrame(combined_data)
df_combined
filtered_df = df_combined[['appId','appDisplayName','viewId','viewName','reportIds','displayName','userPrincipalName','emailAddress']]
filtered_df